In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import scipy.interpolate
import importlib
import modules.loaddata

# Data prep

In [3]:
# This class loads the data and splits it into training and test. It can be configured multiple times for different parameters
class DataManager():
    def __init__(self, USE_RESIDUALS=1):

        self.spectral_q, self.spectral_vals = modules.loaddata.load_spectral("./data/sf_nu_responses/")
        self.ab_initio_q, self.ab_initio_vals = modules.loaddata.load_ab_initio("./data/ab_initio_nu_responses/")
        
        self.spectral_functions = modules.loaddata.build_spectral_interpolator(self.spectral_q, self.spectral_vals)

        self.setup_residuals(USE_RESIDUALS=USE_RESIDUALS)
    
    def setup_residuals(self, USE_RESIDUALS):
        if USE_RESIDUALS is not None:
            self.USE_RESIDUALS = USE_RESIDUALS
        residual = self.ab_initio_vals.copy()
        for (q_idx, q) in enumerate(self.ab_initio_q):
            for (curve_idx, curve) in enumerate(self.ab_initio_vals[q_idx]):
                residual[q_idx, curve_idx, 1] = curve[1] - self.USE_RESIDUALS*self.spectral_functions[curve_idx]((q, curve[0]))
        self.residual = residual

    
    def configure_data(self, q_to_test=[]):
        X_train = []
        Y_train = []
        
        X_test = []
        Y_test = []


        for q_idx, q in enumerate(self.ab_initio_q):
            # TODO: Check that they actually share w values
            w_vals = self.residual[q_idx, 0, 0]
            
            y0 = self.residual[q_idx, 0, 1]
            y1 = self.residual[q_idx, 1, 1]
            y2 = self.residual[q_idx, 2, 1]
            
            for i, w in enumerate(w_vals):
                if not q > w: # skip nonphysical poins
                    continue
                # Skip points where any of the curves resulted in NaN. Spectral functions only cover the valid region q<w, while ab initio extends outside that
                if not (np.isnan(y0[i]) or np.isnan(y1[i]) or np.isnan(y2[i])):
                    x = [w, q]
                    y = [y0[i], y1[i], y2[i]]
                    if q in q_to_test:
                        X_test.append(x)
                        Y_test.append(y)
                    else:
                        X_train.append(x)
                        Y_train.append(y)

        X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
        Y_train = torch.tensor(Y_train, dtype=torch.float32).to(device)
        
        X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
        Y_test = torch.tensor(Y_test, dtype=torch.float32).to(device)

        # Normalized data
        self.X_mean = X_train.mean(dim=0, keepdim=True)
        self.X_std = X_train.std(dim=0, keepdim=True)
        self.Y_mean = Y_train.mean(dim=0, keepdim=True)
        self.Y_std = Y_train.std(dim=0, keepdim=True)

        X_train = self.x_to_normalised(X_train)
        Y_train = self.y_to_normalised(Y_train)
        
        X_test = self.x_to_normalised(X_test)
        Y_test = self.y_to_normalised(Y_test)


        from torch.utils.data import TensorDataset, DataLoader

        train_dataset = TensorDataset(X_train, Y_train)
        train_dataloader = DataLoader(train_dataset, batch_size=20000, shuffle=True)
        
        test_dataset = TensorDataset(X_test, Y_test)
        test_dataloader = DataLoader(test_dataset, batch_size=20000, shuffle=False)

        # print(f"X_train shape: {X_train.shape}")
        # print(f"Y_train shape: {Y_train.shape}")
        self.train_dataloader = train_dataloader
        self.test_dataloader = test_dataloader
    
    def x_to_normalised(self, X):
        X = (X - self.X_mean) / self.X_std 
        return X
    def y_to_normalised(self, Y):
        Y = (Y - self.Y_mean) / self.Y_std
        return Y

    def x_from_normalized(self, X):
        X = X * self.X_std + self.X_mean
        return X
    def y_from_normalized(self, Y):
        Y = Y * self.Y_std + self.Y_mean
        return Y
    

# Torch Setup

In [4]:
from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
if device == "cuda":
    print("CUDA available")
else:
    print("⚠️ CUDA not available ⚠️")

CUDA available


In [5]:
# This class stores the parameter for the network itself.
class NNParams:
    def __init__(self, layers = [1024, 1024, 128], learning_rate = 1e-3):
        self.layers = layers
        self.learning_rate = learning_rate
    
    # Make pretty so it can be printed and logged
    def __str__(self):
        # procedurally print all fields
        res = "NNParams:\n"
        for field in self.__dict__:
            res += f" - {field}: {self.__dict__[field]}\n"
        return res

    def as_file_friendly_str(self):
        res = ""
        for field in self.__dict__:
            res += f"{field}={self.__dict__[field]}\n"
        return res
    

In [6]:
# Loss function used in training
def custom_loss(pred, target):
    return torch.mean(torch.abs(pred - target)*torch.abs(target))

In [7]:
# Create model
class ResidualNN(nn.Module):
    def __init__(self, dataManager: DataManager, nnParams: NNParams):
        super().__init__()
        self.dataManager = dataManager
        self.nnParams = nnParams

        self.flatten = nn.Flatten()
        
        self.linear_relu_stack = nn.Sequential()
        self.linear_relu_stack.add_module("input_layer", nn.Linear(2, nnParams.layers[0]))
        for i, layer_size in enumerate(nnParams.layers[1:], start=1):
            self.linear_relu_stack.add_module(f"layer_{i}", nn.Linear(nnParams.layers[i-1], layer_size))
            self.linear_relu_stack.add_module(f"activation_{i}", nn.SELU())
        self.linear_relu_stack.add_module("output_layer", nn.Linear(nnParams.layers[-1], 3))
        self.loss_fn = custom_loss
        self.optimizer = torch.optim.Adam(self.parameters(), lr=nnParams.learning_rate)

    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
    def train_single_iteration(self, X, y, log: bool): # Only run this when the setup in train_for has been done
        pred = self(X)
        loss = self.loss_fn(pred, y)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        loss = loss.item()
        if log:
            print(f"loss: {loss:>7f} ")
    
    def train_for(self, steps: int, log: bool = True):
        '''
        Train the network on the data provided by the DataManager for the given number of steps.
        - steps: Number of epochs to run for
        - log: Whether to print info during training. Disable to clear up the output
        '''
        self.train()
        X, y = next(iter(self.dataManager.train_dataloader)) # Everything fits in one batch, saves on data transfer to GPU
        X, y = X.to(device), y.to(device)
        for t in range(steps):
            if log:
                print(f"Epoch {t+1}\n-------------------------------")
            self.train_single_iteration(X, y, log)
        if log:
            print("Done!")
    
    def test(self, log: bool = True):
        self.eval()
        X, y = next(iter(self.dataManager.test_dataloader))
        X, y = X.to(device), y.to(device)
        loss = 0
        with torch.no_grad():
            pred = self(X)
            loss = self.loss_fn(pred, y).item()
        if log:
            print(f"Test loss: {loss:>8f} \n")
        return loss
    
    def make_transformed_prediction(self, q, wvals):
        inputs = torch.tensor([[w, q] for w in wvals], dtype=torch.float32).to(device)
        inputs = self.dataManager.x_to_normalised(inputs)
        with torch.no_grad():
            preds = self(inputs)
            preds = self.dataManager.y_from_normalized(preds).cpu().numpy()
        return preds

# Basic usage:
# dataManager.configure_data(q_to_test=[200])
# nnparams = NNParams(layers=[128, 128, 128], learning_rate=1e-3)
# model = ResidualNN(dataManager, nnparams).to(device)
# model.train_for(5000, log=True)

# Parameter sweeping
Below the parameter space is looped over and the error is evaluated for that parameter set. This is done by dropping out on q-curve (except the outermost ones, to avoid teaching extrapolation), training the network on the remaining curves and then computing the error from the dropped curve. Currently only the first curve is used (`Rxy` I think).

Note that the final error is computed in MeV (rather than the arbitrary normalised units used inside the NN). This is because the normalisation might be different for different parameter sets, in particular depending on whether residuals are used or not.

Resulting data is written to `parameter_sweep.txt`. Setting `CLEAR_OUTPUT` clears the file before each run.

In [18]:
CLEAR_OUTPUT = True
if CLEAR_OUTPUT:
    open("parameter_sweep.txt", "w").close() # Clear output file

dataManager = DataManager(USE_RESIDUALS=1)

import itertools
import time

# ---- All parameter values to test ---------
lr_values = [1e-3, 3e-4, 1e-4]
layers_values = [[128, 128], [64, 64, 64]]
use_residuals_values = [1, 0]
# -------------------------------------------

# Produce all combinations of hyperparameters
params_combinations = list(itertools.product(lr_values, layers_values, use_residuals_values))
time_start = time.time()
print(f"Testing {len(params_combinations)} combinations")

AVERAGES = 3

total_iterations = len(params_combinations) * (len(dataManager.ab_initio_q)-2) * AVERAGES

iteration_count = 0


for combination_idx, (lr, layers, use_residuals) in enumerate(params_combinations):
    # Setup parameters for this run
    nnparams = NNParams(layers=layers, learning_rate=lr)
    print(nnparams)
    errors = []
    dataManager.setup_residuals(USE_RESIDUALS=use_residuals)
    for q, ab_initio_vals in zip(dataManager.ab_initio_q[1:-1], dataManager.ab_initio_vals[1:-1]): # Skip dropping first and last q to avoid extrapolation
        dataManager.configure_data(q_to_test=[q]) # Setup the training/test data
        for average_idx in range(AVERAGES): # Take averages

            # Train model
            model = ResidualNN(dataManager, nnparams).to(device)
            model.train_for(5000, log=False)


            # Do evaluation on the dropped curve
            wvals = ab_initio_vals[0, 0]
            physical_mask = wvals < q # Filter out nonphysical points
            wvals = wvals[physical_mask]
            residual_pred = model.make_transformed_prediction(q, wvals) # Make prediction for the dropped curve

            predicted_final = np.nan_to_num(dataManager.spectral_functions[0]((q, wvals))*dataManager.USE_RESIDUALS + residual_pred[:, 0])
            true_final = ab_initio_vals[0, 1][physical_mask]

            # This error is calculated in MeV, not the normalized units used in training, to get an invariant measure of performance
            errors.append(np.mean(np.abs(predicted_final - true_final))) 

            plt.figure()
            plt.plot(wvals, predicted_final, label="predicted", linewidth=0.2)
            plt.plot(wvals, true_final, label="true", linewidth=0.2)
            plt.legend()
            plt.title("q="+str(q)  +" iter: "+str(average_idx))
            plt.layout = "tight"
            filename = f"param_sweep_figures/{combination_idx}_q{q}_{average_idx}.png"
            plt.savefig(filename, dpi=600)
            plt.close()

            # Print progress and ETA
            expected_time = (time.time() - time_start) / (iteration_count + 1) * (total_iterations - iteration_count - 1)
            hours = int(expected_time // 3600)
            minutes = int((expected_time % 3600) // 60)
            seconds = int(expected_time % 60)
            print(f"{(iteration_count + 1)/(total_iterations) * 100:02.2f}% | ETA: {hours:02d}:{minutes:02d}:{seconds:02d}")
            iteration_count += 1

    loss = np.mean(errors)
    dev = np.std(errors)
    with open("parameter_sweep.txt", "a") as f:
        f.write(f"loss={loss}\n")
        f.write(f"dev={dev}\n")
        f.write(f"combination_idx={combination_idx}\n")
        f.write(f"use_residuals={use_residuals}\n")
        f.write(nnparams.as_file_friendly_str())
        f.write("\n")

    print(f"Loss: {loss:.6f} ± {dev:.6f}")

Testing 12 combinations
NNParams:
 - layers: [128, 128]
 - learning_rate: 0.001

0.46% | ETA: 00:10:42
0.93% | ETA: 00:10:31
1.39% | ETA: 00:10:43
1.85% | ETA: 00:10:43
2.31% | ETA: 00:10:54
2.78% | ETA: 00:10:43
3.24% | ETA: 00:10:37
3.70% | ETA: 00:10:31
4.17% | ETA: 00:10:31
4.63% | ETA: 00:10:29
5.09% | ETA: 00:10:29
5.56% | ETA: 00:10:26
6.02% | ETA: 00:10:29
6.48% | ETA: 00:10:26
6.94% | ETA: 00:10:25
7.41% | ETA: 00:10:22
7.87% | ETA: 00:10:15
8.33% | ETA: 00:10:08
Loss: 3.840947 ± 2.097413
NNParams:
 - layers: [128, 128]
 - learning_rate: 0.001

8.80% | ETA: 00:10:05
9.26% | ETA: 00:10:01
9.72% | ETA: 00:09:57
10.19% | ETA: 00:09:52
10.65% | ETA: 00:09:48
11.11% | ETA: 00:09:43
11.57% | ETA: 00:09:58
12.04% | ETA: 00:10:32
12.50% | ETA: 00:11:02
12.96% | ETA: 00:11:32
13.43% | ETA: 00:11:59
13.89% | ETA: 00:12:23
14.35% | ETA: 00:12:40
14.81% | ETA: 00:12:53
15.28% | ETA: 00:13:07
15.74% | ETA: 00:13:23
16.20% | ETA: 00:13:39
16.67% | ETA: 00:13:52
Loss: 4.554445 ± 1.693247
NNP

# The below is for visualising the interpolated curves (don't need to run anything below here)
Note that the model used will be the last one trained in the above loop, so might be terrible

In [ ]:
# Plot separate curves
wvals = np.arange(0, 500, 1)
for idx, q in enumerate(dataManager.ab_initio_q):
    preds = model.make_transformed_prediction(q, wvals)
    
    plt.figure(figsize=(10, 6))
    plt.plot(wvals, preds[:, 0], label="Curve 1", color="red")
    plt.plot(wvals, preds[:, 1], label="Curve 2", color="blue")
    plt.plot(wvals, preds[:, 2], label="Curve 3", color="green")

    # Actual residuals
    plt.plot(dataManager.residual[idx, 0, 0], dataManager.residual[idx, 0, 1], label="Actual Curve 1", linestyle="dashed", color="red")
    plt.plot(dataManager.residual[idx, 1, 0], dataManager.residual[idx, 1, 1], label="Actual Curve 2", linestyle="dashed", color="blue")
    plt.plot(dataManager.residual[idx, 2, 0], dataManager.residual[idx, 2, 1], label="Actual Curve 3", linestyle="dashed", color="green")

    plt.title(f"Predicted Residuals for q={q} MeV")
    plt.xlabel("Energy Transfer (w)")
    plt.ylabel("Residual")
    plt.legend()
    plt.xlim(-10, 200)
    # plt.ylim(-200, 200)
    plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
colors = ["red", "blue", "green"]
for q_index, (ab_initio_index, q) in enumerate(zip([1, 3, 5], [100, 200, 300])):
    wvals = np.arange(0, 500, 1)
    residual_pred = model.make_transformed_prediction(q, wvals)
    predicted_final = dataManager.spectral_functions[0]((q, wvals))*dataManager.USE_RESIDUALS + residual_pred[:, 0]
    plt.plot(wvals, predicted_final, label=f"Interpolated q={q}", color=colors[q_index], linestyle="--")
    plt.plot(dataManager.ab_initio_vals[ab_initio_index,0,0], dataManager.ab_initio_vals[ab_initio_index,0,1], label=f"Ab initio original q={q}", color=colors[q_index])

plt.legend()
plt.xlim(0, 300)
plt.xlabel("omega")
plt.ylabel("Response")

# Testing of DataManager and old preprocessing code (junk)

In [11]:
demoDataManager = DataManager(USE_RESIDUALS=1)

In [12]:
# Maybe reintroduce the peak transformation (Scaling, shifting etc.) but I don't think this will help that much

# import modules.interpolation_preprocess
# import importlib
# importlib.reload(modules.interpolation_preprocess)
# preprocess_transform_params = modules.interpolation_preprocess.get_transform_params(spectral_q, spectral_vals)
# spectral_vals_transformed = modules.interpolation_preprocess.transform_data(spectral_q, spectral_vals, preprocess_transform_params)

In [ ]:
# plt.figure(figsize=(22, 15))
import modules.utils

colors = modules.utils.generate_colourscale(len(demoDataManager.ab_initio_q))
for (q_index, vals) in enumerate(demoDataManager.ab_initio_vals):
    curve_idx = 0
    plt.plot(vals[curve_idx,0], vals[curve_idx,1], color=colors[q_index], label=f"Ab initio q={demoDataManager.ab_initio_q[q_index]}")
    plt.plot(vals[curve_idx,0], vals[curve_idx,2], color=colors[q_index], label=f"Ab initio q={demoDataManager.ab_initio_q[q_index]}")
    plt.plot(vals[curve_idx,0], vals[curve_idx,3], color=colors[q_index], label=f"Ab initio q={demoDataManager.ab_initio_q[q_index]}")
plt.xlim(0, 200)
plt.title("Ab initio data")
plt.legend()

In [ ]:
# Plot residuals
# plt.figure(figsize=(15, 10))
for q_index in range(8):
    plt.plot(demoDataManager.residual[q_index,curve_idx,0], demoDataManager.residual[q_index,curve_idx,1], label=f"$q={demoDataManager.ab_initio_q[q_index]}$")
plt.xlim(0, 150)
plt.title("Residuals")
plt.legend()
pass